# Experiment 6 — Subgroup Fairness Analysis

Evaluates model performance across demographic subgroups:
- **Sex**: male vs female
- **Age group**: <50, 50-65, 65-75, >75
- **Ethnicity**: White, Black, Hispanic, Asian, Other

**Key outputs:**
- Per-subgroup AUROC, sensitivity, specificity
- Equalized Odds difference
- Demographic parity gap
- Love plot of performance disparities

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.evaluation.fairness import evaluate_subgroup_fairness, assess_disparities, plot_fairness_disparities

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)
X_test_p  = pipeline.transform(X_test)

trainer = BaselineModelTrainer(cfg)
results = trainer.train_all(X_train_p, y_train, X_val_p, y_val)
best = results['xgboost']['model']

drop_cols = ['subject_id','cancer_type','gender','age']
Xte = X_test_p.drop(columns=[c for c in drop_cols if c in X_test_p.columns]).fillna(0)
y_prob = best.predict_proba(Xte)[:, 1]
y_prob_series = pd.Series(y_prob, index=y_test.index)

print('Model ready. Evaluating fairness...')

In [ ]:
fairness_df = evaluate_subgroup_fairness(X_test_p, y_test, y_prob_series)
print(fairness_df.to_string())

disparities = assess_disparities(fairness_df)
print('\nDisparities:', disparities)

plot_fairness_disparities(fairness_df, '../results/figures/exp6_fairness.png')
fairness_df.to_csv('../results/exp6_fairness_results.csv', index=False)
print('Saved.')